# EuroSAT: entrainement GPU sur Colab

Ce carnet ne contient aucune logique d'entrainement. Il clone le depot public, installe le paquet
et appelle les memes commandes que la machine locale (`python -m eurosat_mlops.train`).
Prerequis : menu Execution > Modifier le type d'execution > GPU T4.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Adam-Blf/eurosat-mlops/blob/main/notebooks/train_colab.ipynb)

In [ ]:
!git clone --depth 1 https://github.com/Adam-Blf/eurosat-mlops.git
%cd eurosat-mlops
!pip install -q -e ".[train]"

In [ ]:
import torch

assert torch.cuda.is_available(), "Aucun GPU: activez-le dans le type d'execution"
print(torch.cuda.get_device_name(0))

Entrainement sur le jeu EuroSAT complet (27 000 images, split 70/15/15 identique a la version CPU).

In [ ]:
!python -m eurosat_mlops.train --epochs 15 --batch 256 --device cuda --out colab_out/models --results colab_out/results

Quantification int8 de ce modele, avec la meme mesure de precision, de latence et de taille.

In [ ]:
!python -m eurosat_mlops.quantize --model colab_out/models/eurosat_fp32.onnx --out colab_out/models --results colab_out/results

Archive a telecharger : modeles ONNX, statistiques de reference pour la derive, metriques JSON et matrice de confusion.

In [ ]:
!cd colab_out && zip -r ../eurosat_colab_artifacts.zip models results -x "*.pt"
from google.colab import files

files.download("eurosat_colab_artifacts.zip")